# Week 4 — Chunking 전략 비교 실험 (Chunking Experiments)

## 목표
3주차 baseline과 동일한 golden set으로 **chunking 전략 3종을 비교**해, 어떤 전략이 본 도메인(한·영 혼합 유방암 가이드라인)에 가장 잘 맞는지를 RAGAS 3지표로 수치화한다.

선행 노트북: `notebooks/week4_data_analysis.ipynb` (데이터 진단)
선행 문서: `docs/week4_retrospective.md` (전략 선택 근거)

## 비교 전략 (회고에서 확정)
| 전략 | 구성 | 근거 |
|---|---|---|
| **A (Baseline)** | RecursiveCharacterTextSplitter 512/100 | 3주차 그대로 (비교 기준점) |
| **B** | RecursiveCharacterTextSplitter 1000/200 | 한국어 페이지당 토큰 분포(평균 947)에 맞춰 chunk_size 키움 |
| **C** | Noise-cleaned + 언어별 차등 size Recursive (ko 900 / en 512) | 머리말·페이지번호 제거 + 한·영 토큰 밀도 2.4배 차이 반영 |

## 비교 설계 (변수 통제)
- A → B: **chunk_size 효과만** 격리 (동일 parser/embedding/LLM/retriever)
- B → C: 여기에 **노이즈 제거 + 언어별 size** 추가 효과
- embedding/LLM/TOP_K는 baseline과 동일로 고정 (chunking 외 변수 차단)


---
## 1. 경로 / 설정 (CONFIG)

baseline 값은 3주차 노트북과 **똑같이** 고정. 전략 B/C 파라미터만 여기서 조정한다.

In [1]:
from pathlib import Path
import os, json
from dotenv import load_dotenv

# 3주차와 동일한 경로 구조
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"
for p in [DATA_PROCESSED, DATA_EVAL, VECTOR_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# .env 로드 (프로젝트 루트 + 작업 디렉토리)
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 없거나 로드 안 됨"
assert os.environ.get(
    "ANTHROPIC_API_KEY"
), "ANTHROPIC_API_KEY가 .env에 없거나 로드 안 됨"

# ---- baseline 고정값 (3주차와 동일: 생성 gpt-4o-mini / 채점 Claude) ----
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
GEN_MODEL = "gpt-4o-mini"  # 답변 생성: OpenAI
JUDGE_MODEL = "claude-haiku-4-5"  # RAGAS 채점: Anthropic (생성과 다른 계열)
EMBED_DEVICE = "cpu"
TOP_K = 5

# ---- 전략별 chunking 파라미터 ----
STRATEGIES = {
    "A_baseline": {"kind": "recursive", "chunk_size": 512, "chunk_overlap": 100},
    "B_size1000": {"kind": "recursive", "chunk_size": 1000, "chunk_overlap": 200},
    "C_cleaned": {
        "kind": "cleaned",
        "size_by_lang": {"ko": 512, "en": 350, "unknown": 450},
        "overlap_by_lang": {"ko": 80, "en": 53, "unknown": 70}, #overlap은 chunk_size의 15%에 근사한 숫자로 설정
    },
}

# 전략 A는 3주차 결과(week3_ragas_scores.csv)가 있으면 재사용 -> 재생성/재평가 생략
REUSE_WEEK3_FOR_A = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OPENAI key:", "OK" if os.environ.get("OPENAI_API_KEY") else "MISSING")
print("ANTHROPIC key:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "MISSING")
print("strategies:", list(STRATEGIES.keys()))

PROJECT_ROOT: /Users/jian/Documents/rag-agent-portfolio
OPENAI key: OK
ANTHROPIC key: OK
strategies: ['A_baseline', 'B_size1000', 'C_cleaned']


---
## 2. 공통 — PDF 로드 (3주차 로더 재사용)

모든 전략이 동일한 원본 Document에서 출발해야 비교가 공정하다.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import re

manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
else:
    meta_lookup = {}


def load_all_pdfs(pdf_root: Path) -> list:
    all_docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        loader = PyMuPDFLoader(str(pdf_path))
        docs = loader.load()
        extra = meta_lookup.get(pdf_path.name, {})
        for d in docs:
            d.metadata.update(
                {
                    "filename": pdf_path.name,
                    "source_folder": pdf_path.parent.name,
                    "org": extra.get("org", pdf_path.parent.name),
                    "title": extra.get("title", pdf_path.stem),
                    "language": extra.get("language", "unknown"),
                    "doc_type": extra.get("doc_type", "unknown"),
                    "priority": extra.get("priority", "unknown"),
                }
            )
        all_docs.extend(docs)
    return all_docs


documents = load_all_pdfs(DATA_RAW / "pdf")
print(f"로드된 페이지 Document 수: {len(documents)}")


# language 메타가 unknown인 경우 한글 비율로 fallback
def guess_lang(text: str) -> str:
    kr = len(re.findall(r"[\uac00-\ud7a3]", text))
    return "ko" if kr > 20 else "en"


for d in documents:
    if d.metadata.get("language") in (None, "unknown", "?"):
        d.metadata["language"] = guess_lang(d.page_content)

로드된 페이지 Document 수: 796


---
## 3. 노이즈 클린 유틸 (전략 C 전용)

회고에서 지적한 두 문제를 보완한 버전이다.

- **문제 1 (페이지번호 과대집계)**: 숫자 줄 전부를 지우면 표 안 숫자까지 삭제됨 → **페이지 맨위/맨아래 위치**의 숫자 줄만 제거
- **문제 2 (본문 오제거)**: "3페이지 이상" 기준은 본문 bullet까지 지움 → **전체 페이지의 30% 이상에 반복**되는 줄만 머리말/꼬리말로 간주

In [3]:
from collections import Counter

PAGE_NUM_PATTERNS = [
    re.compile(r"^\d{1,4}$"),
    re.compile(r"^-\s?\d{1,4}\s?-$"),
    re.compile(r"^Page\s+\d+", re.IGNORECASE),
    re.compile(r"^\d+\s*/\s*\d+$"),
]


def detect_running_headers(
    doc_pages: list, min_ratio: float = 0.30, max_len: int = 50
) -> set:
    """한 문서 내에서 전체 페이지의 min_ratio 이상에 반복되는 짧은 줄 = 머리말/꼬리말 후보."""
    n_pages = len(doc_pages)
    if n_pages < 4:
        return set()
    counter = Counter()
    for text in doc_pages:
        lines = {l.strip() for l in text.split("\n") if 1 <= len(l.strip()) <= max_len}
        for l in lines:
            counter[l] += 1
    threshold = max(3, int(n_pages * min_ratio))
    return {l for l, c in counter.items() if c >= threshold}


def strip_page_number_lines(text: str) -> str:
    """페이지 맨위/맨아래 위치의 숫자 전용 줄만 제거 (표 안 숫자 보존)."""
    lines = text.split("\n")
    nonempty_idx = [i for i, l in enumerate(lines) if l.strip()]
    if not nonempty_idx:
        return text
    edge = set(nonempty_idx[:2] + nonempty_idx[-2:])  # 앞 2줄, 끝 2줄
    out = []
    for i, l in enumerate(lines):
        s = l.strip()
        if i in edge and any(p.match(s) for p in PAGE_NUM_PATTERNS):
            continue
        out.append(l)
    return "\n".join(out)


def clean_documents(docs: list) -> list:
    """filename 단위로 머리말 감지 -> 머리말·페이지번호 제거."""
    by_file = {}
    for d in docs:
        by_file.setdefault(d.metadata.get("filename", "?"), []).append(d)

    cleaned = []
    total_removed = 0
    for fname, pages in by_file.items():
        headers = detect_running_headers([p.page_content for p in pages])
        for d in pages:
            text = strip_page_number_lines(d.page_content)
            kept = []
            for l in text.split("\n"):
                if l.strip() in headers:
                    total_removed += 1
                    continue
                kept.append(l)
            cleaned.append(
                Document(page_content="\n".join(kept), metadata=dict(d.metadata))
            )
    print(f"제거된 머리말/꼬리말 줄 수(누적): {total_removed}")
    return cleaned


# 감지된 머리말 샘플 확인 (ESMO)
sample_pages = [
    d.page_content
    for d in documents
    if d.metadata.get("filename", "").startswith("esmo")
]
print("ESMO 머리말/꼬리말 후보:", detect_running_headers(sample_pages))

ESMO 머리말/꼬리말 후보: {'환자를 위한 ESMO 안내서', '유방암'}


---
## 4. 청킹 전략 3종 정의

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def make_splitter(size: int, overlap: int) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=SEPARATORS,
        length_function=len,
    )


def build_chunks(strategy_name: str) -> list:
    cfg = STRATEGIES[strategy_name]
    if cfg["kind"] == "recursive":
        return make_splitter(cfg["chunk_size"], cfg["chunk_overlap"]).split_documents(
            documents
        )
    if cfg["kind"] == "cleaned":
        cleaned = clean_documents(documents)
        out = []
        for lang in set(d.metadata.get("language", "unknown") for d in cleaned):
            size = cfg["size_by_lang"].get(lang, cfg["size_by_lang"]["unknown"])
            overlap = cfg["overlap_by_lang"].get(
                lang, cfg["overlap_by_lang"]["unknown"]
            )
            sub = [d for d in cleaned if d.metadata.get("language", "unknown") == lang]
            out.extend(make_splitter(size, overlap).split_documents(sub))
        return out
    raise ValueError(strategy_name)


# 각 전략 chunk 통계 (인덱싱 전, 빠름)
chunk_store = {}
for name in STRATEGIES:
    ck = build_chunks(name)
    chunk_store[name] = ck
    avg = sum(len(c.page_content) for c in ck) / max(len(ck), 1)
    print(f"{name:12s} chunk수={len(ck):5d}  평균길이(문자)={avg:.0f}")

A_baseline   chunk수= 3214  평균길이(문자)=436
B_size1000   chunk수= 1753  평균길이(문자)=790
제거된 머리말/꼬리말 줄 수(누적): 1560
C_cleaned    chunk수= 3582  평균길이(문자)=360


---
## 5. 인덱싱 / retriever 빌더 (전략별 별도 컬렉션)

In [5]:
import os
from huggingface_hub import snapshot_download

# 이전 실행에서 켜진 오프라인 모드 해제 + 네트워크 확인 타임아웃 (임베딩 hang 방지)
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# 로컬 캐시에 완전한 snapshot이 있으면 그 경로를 직접 사용 (repo_id 조회로 인한 hang 회피)
try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
    print("로컬 캐시에서 모델 snapshot 확인:", model_dir)
except Exception as e:
    print("로컬 캐시 불완전 -> 다운로드 진행:", repr(e))
    model_dir = snapshot_download(
        repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1
    )
    print("모델 다운로드 완료:", model_dir)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(
    model_name=model_dir,  # repo_id가 아니라 실제 로컬 snapshot 경로 사용
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)
print("embedding 로드 완료")


def build_retriever(strategy_name: str):
    """전략별 Chroma 컬렉션 생성(없으면) 또는 재사용."""
    vdir = VECTOR_ROOT / f"week4_{strategy_name}"
    vdir.mkdir(parents=True, exist_ok=True)
    coll = f"breast_rag_week4_{strategy_name}"
    client = chromadb.PersistentClient(path=str(vdir))
    if coll in [c.name for c in client.list_collections()]:
        vs = Chroma(
            collection_name=coll,
            embedding_function=embeddings,
            persist_directory=str(vdir),
        )
        print(f"  {strategy_name}: 기존 컬렉션 재사용 ({vs._collection.count()}개)")
    else:
        ck = chunk_store[strategy_name]
        print(f"  {strategy_name}: 신규 인덱싱 {len(ck)}개 ... (CPU면 시간 소요)")
        vs = Chroma.from_documents(
            documents=ck,
            embedding=embeddings,
            collection_name=coll,
            persist_directory=str(vdir),
        )
    return vs.as_retriever(search_kwargs={"k": TOP_K})

로컬 캐시에서 모델 snapshot 확인: /Users/jian/.cache/huggingface/hub/models--intfloat--multilingual-e5-base/snapshots/d128750597153bb5987e10b1c3493a34e5a4502a
embedding 로드 완료


---
## 6. RAG 체인 (3주차와 동일 프롬프트)

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]"""
)


def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parts.append(
            f"[{i}] 출처: {m.get('org','?')} / {m.get('title','?')} / p.{m.get('page','?')}\n{d.page_content}"
        )
    return "\n\n---\n\n".join(parts)


def make_ask(retriever):
    def ask(question: str):
        docs = retriever.invoke(question)
        prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
        answer = (llm | StrOutputParser()).invoke(prompt)
        return {
            "question": question,
            "answer": answer,
            "contexts": [d.page_content for d in docs],
        }

    return ask

---
## 7. 평가 질문 세트 (3주차 golden_set_v0 재사용)

3주차와 **동일한 질문 세트**를 써야 baseline 비교가 성립한다. (필요시 20개로 확장 — 5주차 golden set 고도화 때)

In [7]:
import pandas as pd

golden_path = DATA_EVAL / "golden_set_v0.csv"
if golden_path.exists():
    df_golden = pd.read_csv(golden_path)
    print(f"golden_set_v0 로드: {len(df_golden)}문항")
else:
    raise FileNotFoundError(
        "golden_set_v0.csv 없음 — 3주차 노트북 Cell 19를 먼저 실행해 생성하세요."
    )
golden = df_golden.to_dict("records")
df_golden[["question"]]

golden_set_v0 로드: 10문항


,question
0,유방암 검진은 몇 살부터 받는 것이 권장되나요?
1,유방암의 주요 위험 요인은 무엇인가요?
2,HER2 양성 유방암이란 무엇인가요?
3,유방암 1기와 2기의 차이는 무엇인가요?
4,DCIS(상피내암)는 침윤성 유방암과 어떻게 다른가요?
5,항호르몬제 치료는 어떤 환자에게 사용되나요?
6,유방절제술 후 재건수술은 어떤 옵션이 있나요?
7,BRCA 유전자 검사는 누구에게 권장되나요?
8,전이성 유방암(4기)의 일반적인 치료 목표는 무엇인가요?
9,유방암 환자가 식이요법에서 주의할 점은?


---
## 8. 전략별 실행 + RAGAS 평가

주의: qwen3:4b CPU 기준, 전략 1개당 생성 10문항 + RAGAS 평가로 수십 분 소요. B/C 두 전략만 돌려도 1시간 이상 걸릴 수 있다.
- 전략 A는 `REUSE_WEEK3_FOR_A=True`면 3주차 점수(`week3_ragas_scores.csv`)를 그대로 사용 → 재생성/재평가 생략
- GPU가 있으면 CONFIG의 `EMBED_DEVICE="cuda"`로 인덱싱 시간을 크게 줄일 수 있다

In [8]:
import nest_asyncio

nest_asyncio.apply()  # Jupyter async 충돌로 인한 RAGAS NaN 방지

from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm
from langchain_anthropic import ChatAnthropic

ragas_llm = LangchainLLMWrapper(ChatAnthropic(model=JUDGE_MODEL, temperature=0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]


def run_strategy(name: str) -> pd.DataFrame:
    retriever = build_retriever(name)
    ask = make_ask(retriever)
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        r = ask(g["question"])
        rows.append(
            {
                "question": r["question"],
                "answer": r["answer"],
                "contexts": r["contexts"],
                "ground_truth": g.get("ground_truth", ""),
            }
        )
    # ragas 0.2 스키마: user_input / response / retrieved_contexts / reference
    # (옛 question/answer/contexts/ground_truth로 넣으면 reference 미전달 -> context_precision=0)
    samples = [
        {
            "user_input": r["question"],
            "response": r["answer"],
            "retrieved_contexts": r["contexts"],
            "reference": r["ground_truth"],
        }
        for r in rows
    ]
    ds = EvaluationDataset.from_list(samples)
    scores = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb)
    df = scores.to_pandas()
    df.to_csv(
        DATA_PROCESSED / f"week4_ragas_{name}.csv", index=False, encoding="utf-8-sig"
    )
    return df


score_tables = {}
for name in STRATEGIES:
    if name == "A_baseline" and REUSE_WEEK3_FOR_A:
        wk3 = DATA_PROCESSED / "week3_ragas_scores.csv"
        if wk3.exists():
            score_tables[name] = pd.read_csv(wk3)
            print(f"{name}: 3주차 결과 재사용 ({wk3.name})")
            continue
        print(f"{name}: week3 결과 없음 -> 직접 실행")
    score_tables[name] = run_strategy(name)
    print(f"{name}: 완료")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


A_baseline: 3주차 결과 재사용 (week3_ragas_scores.csv)
  B_size1000: 신규 인덱싱 1753개 ... (CPU면 시간 소요)


RAG[B_size1000]:   0%|          | 0/10 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
RAG[B_size1000]: 100%|██████████| 10/10 [00:28<00:00,  2.85s/it]


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[6]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[18]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Exception raised in Job[3]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


B_size1000: 완료
  C_cleaned: 신규 인덱싱 3582개 ... (CPU면 시간 소요)


RAG[C_cleaned]: 100%|██████████| 10/10 [00:29<00:00,  2.96s/it]


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[3]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


C_cleaned: 완료


---
## 9. 비교 결과표

In [9]:
rows = []
for name in STRATEGIES:
    df = score_tables[name]
    ck = chunk_store[name]
    row = {
        "전략": name,
        "chunk수": len(ck),
        "평균길이": round(sum(len(c.page_content) for c in ck) / max(len(ck), 1)),
    }
    for col in METRIC_COLS:
        row[col] = round(df[col].mean(), 4) if col in df.columns else None
    rows.append(row)

df_compare = pd.DataFrame(rows)

# baseline 대비 델타 추가
base = df_compare[df_compare["전략"] == "A_baseline"].iloc[0]
for col in METRIC_COLS:
    df_compare[col + "_delta"] = (df_compare[col] - base[col]).round(4)

df_compare.to_csv(
    DATA_PROCESSED / "week4_chunking_comparison.csv", index=False, encoding="utf-8-sig"
)
print("저장: week4_chunking_comparison.csv")
df_compare

저장: week4_chunking_comparison.csv


,전략,chunk수,평균길이,faithfulness,answer_relevancy,context_precision,faithfulness_delta,answer_relevancy_delta,context_precision_delta
0,A_baseline,3214,436,0.6480,0.8704,0.7428,0.0000,0.0000,0.0000
1,B_size1000,1753,790,0.5322,0.7841,0.6019,-0.1158,-0.0863,-0.1409
2,C_cleaned,3582,360,0.6020,0.8092,0.7060,-0.0460,-0.0612,-0.0368


---
## 10. 에러 케이스 분석 (전략별 최저 context_precision 질문)

In [10]:
for name in STRATEGIES:
    df = score_tables[name]
    if "context_precision" not in df.columns:
        continue
    # 최신 RAGAS 컬럼명 대응
    COL_Q = "user_input" if "user_input" in df.columns else "question"
    COL_C = "retrieved_contexts" if "retrieved_contexts" in df.columns else "contexts"
    print("=" * 64)
    print(f"[{name}] context_precision 최저 2문항")
    worst = df.nsmallest(2, "context_precision")
    for _, r in worst.iterrows():
        fp = r.get("faithfulness", float("nan"))
        print(f"  질문: {r[COL_Q]}")
        print(
            f"  context_precision={r['context_precision']:.3f}  faithfulness={fp:.3f}"
        )
        ctxs = r[COL_C] if isinstance(r[COL_C], list) else []
        for i, c in enumerate(ctxs[:2], 1):
            print(f"    [{i}] {str(c)[:150].replace(chr(10),' ')} ...")
        print()

[A_baseline] context_precision 최저 2문항
  질문: 유방암 1기와 2기의 차이는 무엇인가요?
  context_precision=0.000  faithfulness=nan

  질문: BRCA 유전자 검사는 누구에게 권장되나요?
  context_precision=0.478  faithfulness=0.700

[B_size1000] context_precision 최저 2문항
  질문: 유방암 1기와 2기의 차이는 무엇인가요?
  context_precision=0.000  faithfulness=0.400
    [1] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 41 제2장 조기 유방암 제3장 재발 및 전이성 유방암 제4장 유전성 유방암  제1장 비침습 유방암 제1장 비침습 유방암: 관상피내암과 소엽상피 ...
    [2] 7 환자를 위한 ESMO 안내서 유방암이란?  유방암은 유방 조직에서 형성되는 암입니다. 일반적으로 유관 (젖을 유두로 운반하는 관) 또는 소엽 (젖을 만드는 샘) 에서 형성됩니다. 유방암은 남성에게서는 드물지만 남성과 여성 모두에서  발생합니다. 여성 유방 해부학.  ...

  질문: 유방암 환자가 식이요법에서 주의할 점은?
  context_precision=0.000  faithfulness=0.556
    [1] 58 유방암 건강 관리 유방암 치료를 받은 후에는 매우 피곤하고 감정적이 될 수 있습니다. 신체가 회복할 시간을 가지면서  충분한 휴식을 취하십시오. 하지만 기분이 좋다면 활동을 제한할 이유가 없습니다. 자신을 잘  돌보면서 가족 활동, 일 또는 직업적 역할을 포함한  ...
    [2] 19.5%, 22.3%)이나 유방보존술(70.1%, 68.4%, 65.3%) 면에 있어 차이가 없어, anthracycline-taxane요법 의 기간을 증가시키거나 capecitabine을 추가하는 것은 권고되지 않는다(